In [1]:
import torch
import triton
import triton.language as tl

from triton.testing import do_bench

print(torch.__version__, triton.__version__, torch.cuda.get_device_name())

2.14.1+cu130 3.8.0 NVIDIA GeForce RTX 3060


In [6]:
@triton.jit
def reduce_sum_kernel(
    x_ptr,
    x_row_stride,
    out_ptr,
    out_row_stride,
    d,
    BLOCK_D: tl.constexpr,
):
    pid1 = tl.program_id(0)
    pid2 = tl.program_id(1)

    col_offsets = pid2 * BLOCK_D + tl.arange(0, BLOCK_D)
    x = tl.load(x_ptr + pid1 * x_row_stride + col_offsets, mask=col_offsets < d, other=0.0)
    local_sum = tl.sum(x, axis=0)

    tl.atomic_add(out_ptr + pid1 * out_row_stride, local_sum)

def reduce_sum(x):
    assert x.ndim == 2

    out = torch.zeros_like(x[:, 0])

    n, d = x.shape
    BLOCK_D = 1024
    grid = (n, triton.cdiv(d, BLOCK_D))

    reduce_sum_kernel[grid](x, x.stride(0), out, out.stride(0), d, BLOCK_D)

    return out


x = torch.arange(17 * 300, device="cuda").reshape(17, 300) / 100

out = reduce_sum(x)

In [8]:
# Benchmark over triton, compile and eager
n = 1024

naive = lambda: torch.sum(x, dim=-1)
yours = lambda: reduce_sum(x)

# torch.testing.assert_close(yours(), eager())

print(f"{"d":8s} {"naive":14s} {"yours":14s}")
for d in [128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072]:
    # torch._dynamo.reset()
    # compiled_fn = torch.compile(lambda x: torch.softmax(x, dim=-1), dynamic=False)
    # compiled = lambda: compiled_fn(x)

    x = torch.randn(n, d).cuda()
    ms_naive = do_bench(naive, warmup=50, rep=1000)
    ms_triton = do_bench(yours, warmup=50, rep=1000)
    print(f"{d:<8d} {ms_naive:<14.3f} {ms_triton:<14.3f}")



d        naive          yours         
128      0.007          0.008         
256      0.009          0.010         
512      0.013          0.013         
1024     0.020          0.021         
2048     0.033          0.034         
4096     0.058          0.059         
8192     0.107          0.109         
16384    0.205          0.209         
32768    0.401          0.408         
65536    0.793          0.806         
131072   1.577          1.603         
